# 1. Insertion Sort Algorithm (Shift-and-insert)

In [1]:
from typing import List

class Pair:
    def __init__(self, key: int, value: str):
        self.key = key
        self.value = value

    def __repr__(self) -> str:
        return f"Pair(key={self.key}, value={self.value!r})"

def insertion_sort(pairs: List[Pair]) -> List[Pair]:
    for i in range(1, len(pairs)):
        current = pairs[i]
        j = i - 1
        # Stable: maintain relative order of equal elements. 
        # This is not in specification of insertion sort, but it is still an invariant of the algorithm?
        while j >= 0 and pairs[j].key > current.key: 
            
            pairs[j + 1] = pairs[j]
            j -= 1
        pairs[j + 1] = current
    return pairs

Using `>` rather than `>=` preserves the relative order of equal-key objects,
so this implementation is stable.

$$\forall x,y.\;
Processed(x)\land Processed(y)\land
OriginalBefore(x,y)\land
KeyEqual(x,y)
\Rightarrow Before(x,y)$$

In [2]:
pair = Pair(5, "apple")
repr(pair)

"Pair(key=5, value='apple')"

In [3]:
def get_keys(pairs: List[Pair]) -> List[int]:
    return [pair.key for pair in pairs]

def get_values(pairs: List[Pair]) -> List[str]:
    return [pair.value for pair in pairs]

assert get_keys(insertion_sort([])) == []
assert get_keys(insertion_sort([Pair(1, "a")])) == [1]
assert get_keys(
    insertion_sort([
        Pair(5, "apple"),
        Pair(2, "banana"),
        Pair(9, "cherry"),
    ])
) == [2, 5, 9]
assert get_keys(
    insertion_sort([
        Pair(4, "a"),
        Pair(3, "b"),
        Pair(2, "c"),
        Pair(1, "d"),
    ])
) == [1, 2, 3, 4]

In [4]:
result = insertion_sort([
    Pair(2, "first"),
    Pair(1, "middle"),
    Pair(2, "second"),
])
assert get_keys(result) == [1, 2, 2]
assert get_values(result) == ["middle", "first", "second"]

print("All insertion-sort tests passed.")

All insertion-sort tests passed.


# 2. Record raw snapshots

In [5]:
from typing import Tuple


def insertion_sort_with_snapshots(
    pairs: List[Pair],
) -> Tuple[List[Pair], List[List[Pair]]]:

    if not pairs:
        return pairs, []

    snapshots: List[List[Pair]] = []

    # Record the initial state before the first outer-loop iteration
    snapshots.append(pairs.copy())


    for i in range(1, len(pairs)):
        current = pairs[i]
        j = i - 1
        # Stable: maintain relative order of equal elements. 
        # This is not in specification of insertion sort, but it is still an invariant of the algorithm?
        while j >= 0 and pairs[j].key > current.key:         
            pairs[j + 1] = pairs[j]
            j -= 1
        pairs[j + 1] = current

        # Record the state after each outer-loop iteration
        snapshots.append(pairs.copy())

    return pairs, snapshots

In [6]:
# Quick Test
pairs, snapshots = insertion_sort_with_snapshots([
        Pair(5, "apple"),
        Pair(2, "banana"),
        Pair(9, "cherry"),
        Pair(1, "pear"),
        Pair(3, "grape")
    ])
snapshots

[[Pair(key=5, value='apple'),
  Pair(key=2, value='banana'),
  Pair(key=9, value='cherry'),
  Pair(key=1, value='pear'),
  Pair(key=3, value='grape')],
 [Pair(key=2, value='banana'),
  Pair(key=5, value='apple'),
  Pair(key=9, value='cherry'),
  Pair(key=1, value='pear'),
  Pair(key=3, value='grape')],
 [Pair(key=2, value='banana'),
  Pair(key=5, value='apple'),
  Pair(key=9, value='cherry'),
  Pair(key=1, value='pear'),
  Pair(key=3, value='grape')],
 [Pair(key=1, value='pear'),
  Pair(key=2, value='banana'),
  Pair(key=5, value='apple'),
  Pair(key=9, value='cherry'),
  Pair(key=3, value='grape')],
 [Pair(key=1, value='pear'),
  Pair(key=2, value='banana'),
  Pair(key=3, value='grape'),
  Pair(key=5, value='apple'),
  Pair(key=9, value='cherry')]]

In [7]:
example = [
    Pair(5, "apple"),
    Pair(2, "banana"),
    Pair(9, "cherry"),
]

sorted_pairs, snapshots = insertion_sort_with_snapshots(example)

snapshot_keys = [get_keys(snapshot) for snapshot in snapshots]

assert snapshot_keys == [
    [5, 2, 9],  # initial state: prefix length 1
    [2, 5, 9],  # after inserting key 2: prefix length 2
    [2, 5, 9],  # after inserting key 9: prefix length 3
]

assert get_keys(sorted_pairs) == [2, 5, 9]
assert snapshots[0] is not snapshots[1]
assert snapshots[1] is not snapshots[2]

print(snapshot_keys)

[[5, 2, 9], [2, 5, 9], [2, 5, 9]]


In [8]:
[id(pair) for pair in example]

[4460378336, 4460379872, 4460378432]

In [9]:
[(prefix_length, current_order)for prefix_length, current_order in enumerate(
        snapshots,
        start=1,
    )]

[(1,
  [Pair(key=5, value='apple'),
   Pair(key=2, value='banana'),
   Pair(key=9, value='cherry')]),
 (2,
  [Pair(key=2, value='banana'),
   Pair(key=5, value='apple'),
   Pair(key=9, value='cherry')]),
 (3,
  [Pair(key=2, value='banana'),
   Pair(key=5, value='apple'),
   Pair(key=9, value='cherry')])]

## 2.1 Stable Object Identity

In [10]:
pairs = [
    Pair(5, "apple"),
    Pair(2, "banana"),
    Pair(9, "cherry"),
]

object_names = {
    id(pair): f"item_{index}"
    for index, pair in enumerate(pairs)
}

keys = {
    object_names[id(pair)]: pair.key
    for pair in pairs
}

values = {
    object_names[id(pair)]: pair.value
    for pair in pairs
}

initial_order = tuple(object_names[id(pair)] for pair in pairs)

print(initial_order)
print(keys)
print(values)

('item_0', 'item_1', 'item_2')
{'item_0': 5, 'item_1': 2, 'item_2': 9}
{'item_0': 'apple', 'item_1': 'banana', 'item_2': 'cherry'}


## 2.2 Define structured snapshot

In [11]:
from dataclasses import dataclass
from typing import Dict, Tuple

@dataclass(frozen=True)
class InsertionSortSnapshot:
    outer_index: int # length of the current sorted prefix
    order: Tuple[str, ...] # the order of the elements in the list at this point in time, e.g., ("item_1", "item_0", "item_2")
    processed: frozenset[str] # objects currently belonging to that sorted prefix, e.g., frozenset({"item_0", "item_1"})
    keys: Dict[str, int] # a mapping from each element to its key, e.g., {"item_0": 5, "item_1": 2, "item_2": 9}
    values: Dict[str, str] # a mapping from each element to its value, e.g., {"item_0": "apple", "item_1": "banana", "item_2": "cherry"}

## 2.3 Tracer

In [12]:
def trace_insertion_sort(pairs: List[Pair]) -> Tuple[List[Pair], Tuple[InsertionSortSnapshot, ...]]:

    working = list(pairs)

    object_ids = [id(pair) for pair in working]

    if len(set(object_ids)) != len(working):
        raise ValueError("Each input position must contain a distinct Pair object.")

    # Stable names are assigned according to the original input positions.
    object_names = {
        id(pair): f"item_{index}" 
        for index, pair in enumerate(working)
    }

    keys = {
        object_names[id(pair)]: pair.key
        for pair in working
    }

    values = {
        object_names[id(pair)]: pair.value
        for pair in working
    }

    sorted_pairs, raw_snapshots = insertion_sort_with_snapshots(working)

    snapshots: List[InsertionSortSnapshot] = []

    for prefix_length, current_order in enumerate(
        raw_snapshots,
        start=1,
    ):
        # TODO 1:
        # Convert the current Pair order into stable object names.
        named_order = tuple(object_names[id(pair)] for pair in current_order)

        # TODO 2:
        # The first prefix_length objects have been processed.
        processed = frozenset(named_order[:prefix_length])

        # TODO 3:
        # Construct and append an InsertionSortSnapshot.
        snapshot = InsertionSortSnapshot(
            outer_index=prefix_length,
            order=named_order,
            processed=processed,
            keys=dict(keys),
            values=dict(values),
        )

        snapshots.append(snapshot)

    return sorted_pairs, tuple(snapshots)

In [13]:
original = [
    Pair(5, "apple"),
    Pair(2, "banana"),
    Pair(9, "cherry"),
]

sorted_pairs, snapshots = trace_insertion_sort(original)

assert get_keys(original) == [5, 2, 9]
assert get_keys(sorted_pairs) == [2, 5, 9]

assert [snapshot.order for snapshot in snapshots] == [
    ("item_0", "item_1", "item_2"),  # initial state: prefix length 1
    ("item_1", "item_0", "item_2"),  # after inserting key 2: prefix length 2
    ("item_1", "item_0", "item_2"),  # after inserting key 9: prefix length 3
]

assert [snapshot.processed for snapshot in snapshots] == [
    frozenset({"item_0"}),  # initial state: prefix length 1
    frozenset({"item_0", "item_1"}),  # after inserting key 2: prefix length 2
    frozenset({"item_0", "item_1", "item_2"}),  # after inserting key 9: prefix length 3
]

assert snapshots[1].keys == {
    "item_0": 5,
    "item_1": 2,
    "item_2": 9,
}

for snapshot in snapshots:
    print(
        "prefix_length =", snapshot.outer_index,
        "order =", snapshot.order,
        "processed =", snapshot.processed,
    )

prefix_length = 1 order = ('item_0', 'item_1', 'item_2') processed = frozenset({'item_0'})
prefix_length = 2 order = ('item_1', 'item_0', 'item_2') processed = frozenset({'item_1', 'item_0'})
prefix_length = 3 order = ('item_1', 'item_0', 'item_2') processed = frozenset({'item_2', 'item_1', 'item_0'})


# 3. Build the Relational Model

## 3.1 Adapt to `RelationalSnapshot`

In [14]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from synthesis.inference_lib.relational import RelationalSnapshot

def to_relational_snapshot(snapshot: InsertionSortSnapshot) -> RelationalSnapshot:

    return RelationalSnapshot(
        # TODO 1:
        # The complete stable object universe
        objects=tuple(snapshot.keys.keys()),

        # TODO 2:
        # Preserve the algorithm-specific state
        payload=snapshot,
    )

In [15]:
relational_snapshot = to_relational_snapshot(snapshots[1])

assert relational_snapshot.objects == ("item_0", "item_1", "item_2")

assert relational_snapshot.payload is snapshots[1]
assert relational_snapshot.constant_bindings == {}

print("objects:", relational_snapshot.objects)
print("current order:", relational_snapshot.payload.order)
print("processed:", relational_snapshot.payload.processed)

objects: ('item_0', 'item_1', 'item_2')
current order: ('item_1', 'item_0', 'item_2')
processed: frozenset({'item_1', 'item_0'})


In [16]:
relational_snapshots = tuple(to_relational_snapshot(snapshot) for snapshot in snapshots)

assert len(relational_snapshots) == len(snapshots)
assert all(
    relational_snapshot.objects == ("item_0", "item_1", "item_2")
    for relational_snapshot in relational_snapshots
)

print(relational_snapshots[1])

RelationalSnapshot(objects=('item_0', 'item_1', 'item_2'), payload=InsertionSortSnapshot(outer_index=2, order=('item_1', 'item_0', 'item_2'), processed=frozenset({'item_1', 'item_0'}), keys={'item_0': 5, 'item_1': 2, 'item_2': 9}, values={'item_0': 'apple', 'item_1': 'banana', 'item_2': 'cherry'}), constant_bindings={})


In [17]:
tuple(snapshots[1].keys)

('item_0', 'item_1', 'item_2')

## 3.2 Define Concrete Relation Evaluators

In [18]:
def processed(
    snapshot: RelationalSnapshot,
    arguments: Tuple[str, ...],
) -> bool:

    (item,) = arguments

    # TODO:
    # Is item in the processed prefix?

    return item in snapshot.payload.processed

In [19]:
second_state = relational_snapshots[1]

assert processed(second_state, ("item_0",)) is True
assert processed(second_state, ("item_1",)) is True
assert processed(second_state, ("item_2",)) is False

In [20]:
def before(
    snapshot: RelationalSnapshot,
    arguments: Tuple[str, ...],
) -> bool:

    left, right = arguments

    positions = {item: index for index, item in enumerate(snapshot.payload.order)}

    return positions[left] < positions[right]

In [21]:
assert before(second_state, ("item_1", "item_0")) is True
assert before(second_state, ("item_0", "item_1")) is False

# Strict order: no object is before itself.
assert before(second_state, ("item_0", "item_0")) is False

In [22]:
assert before(relational_snapshots[0], ("item_0", "item_1")) is True
assert before(relational_snapshots[1], ("item_0", "item_1")) is False

In [23]:
def key_le(
    snapshot: RelationalSnapshot,
    arguments: Tuple[str, ...],
) -> bool:

    left, right = arguments

    return snapshot.payload.keys[left] <= snapshot.payload.keys[right]

In [24]:
assert key_le(second_state, ("item_1", "item_0")) is True   # 2 <= 5
assert key_le(second_state, ("item_0", "item_1")) is False  # 5 <= 2
assert key_le(second_state, ("item_0", "item_0")) is True   # 5 <= 5

In [25]:
for state in relational_snapshots:
    assert key_le(state, ("item_1", "item_0")) is True

In [26]:
relation_results = {
    "Processed(item_1)": processed(second_state, ("item_1",)),
    "Processed(item_2)": processed(second_state, ("item_2",)),
    "Before(item_1,item_0)": before(
        second_state, ("item_1", "item_0")
    ),
    "KeyLE(item_1,item_0)": key_le(
        second_state, ("item_1", "item_0")
    ),
}

assert relation_results == {
    "Processed(item_1)": True,
    "Processed(item_2)": False,
    "Before(item_1,item_0)": True,
    "KeyLE(item_1,item_0)": True,
}

relation_results

{'Processed(item_1)': True,
 'Processed(item_2)': False,
 'Before(item_1,item_0)': True,
 'KeyLE(item_1,item_0)': True}

## 3.3 Decribe Relations with `RelationSpec`

In [27]:
from synthesis.inference_lib.relational import (
    RelationSpec,
    RelationalDomain,
)

processed_spec = RelationSpec(
    name="Processed",       # symbolic predicate name
    arity=1,      # number of object arguments
    evaluator=processed,  # concrete Python function
)

before_spec = RelationSpec(
    name="Before",
    arity=2,
    evaluator=before,
)

key_le_spec = RelationSpec(
    name="KeyLE",
    arity=2,
    evaluator=key_le,
)

In [28]:
assert processed_spec.name == "Processed"
assert processed_spec.arity == 1
assert processed_spec.evaluator is processed

assert before_spec.name == "Before"
assert before_spec.arity == 2
assert before_spec.evaluator is before

assert key_le_spec.name == "KeyLE"
assert key_le_spec.arity == 2
assert key_le_spec.evaluator is key_le

In [29]:
assert processed_spec.evaluator(
    second_state,
    ("item_1",),
) is True

## 3.4 Build minimal `RelationalDomain`

In [36]:
domain = RelationalDomain(
    name="InsertionSortItem",
    relation_specs=(
        processed_spec,
        before_spec,
        key_le_spec,
    ),
    include_equality=True,
)

In [ ]:
print("Object sort:", domain.object_sort) 
# object_sort is the Z3 type inhabited by insertion-sort objects.
# The concrete finite object universe is snapshot.objects.

for name, relation in domain.relations.items():
    print(name, "->", relation)

Object sort: InsertionSortItem
Processed -> Processed
Before -> Before
KeyLE -> KeyLE


In [ ]:
x = domain.constant("x")
y = domain.constant("y")

# These are z3 expressions, not Python booleans.
print(domain.relation("Processed")(x))
print(domain.relation("Before")(x, y))
print(domain.relation("KeyLE")(x, y))

Processed(x)
Before(x, y)
KeyLE(x, y)


In [43]:
import z3

assert isinstance(
    domain.relation("Before")(x, y),
    z3.BoolRef,
)

In [46]:
assert domain.evaluate(
    "Processed",
    second_state,
    ("item_1",),
) is True

assert domain.evaluate(
    "Before",
    second_state,
    ("item_1", "item_0"),
) is True

assert domain.evaluate(
    "KeyLE",
    second_state,
    ("item_1", "item_0"),
) is True

In [47]:
try:
    domain.evaluate(
        "Processed",
        second_state,
        ("item_0", "item_1"),
    )
except ValueError as error:
    print(error)

Relation 'Processed' expects 1 arguments, got 2.


# 4. Add Domain Axioms  

`Before` models the current positional order and is a strict total order:

1. Irreflexive:
   $\forall x.\ \neg Before(x,x)$

2. Transitive:
   $\forall x,y,z.\ Before(x,y)\land Before(y,z)
   \Rightarrow Before(x,z)$

3. Total on distinct objects:
   $\forall x,y.\ x=y\lor Before(x,y)\lor Before(y,x)$

`KeyLE` models `key(x) <= key(y)` and is a total preorder:

4. Reflexive:
   $\forall x.\ KeyLE(x,x)$

5. Transitive:
   $\forall x,y,z.\ KeyLE(x,y)\land KeyLE(y,z)
   \Rightarrow KeyLE(x,z)$

6. Total:
   $\forall x,y.\ KeyLE(x,y)\lor KeyLE(y,x)$

# 5. Build the Symbolic Vocabulary

# 6. Ground Concrete Truth Rows

# 7. Build the Training Corpus

# 8. Call the Core API

# 9. Interpret and Validate Results